In [1]:
import pandas as pd

SUBCATEGORY_KEYWORDS = {
    # Geology
    "volcano": [
        "volcanic plume", "lava", "eruption", "ash plume",
        'Was the nighttime brightness over northern part of Isabela Island greater than the southern part of Isabela Island at 1:20 am local time on January 7, 2022?'.lower(), 
        'Was the nighttime brightness over Wolf Volcano on Isabela Island brighter on January 7, 2022 then January 6 2022?'.lower(),
        "Jabal Qidr have a darker color than Jabal Bayda".lower()
    ],
    "landform_change": [
        "island", "shoreline", "erosion",
    ],
    "extreme_weather": [
        "cyclone",
    ],
    
    # Atmosphere
    "trace_gasses": [
        "no2", "nitrogen dioxide", "nitrogen oxide", "carbon monoxide", "ozone", "ozone hole",
    ],
    "aerosals": [
        "aerosol optical depth", "black carbon", "pm2.5",
    ],
    "clouds_fog": [
        "cloud", "fog", "Mount Vesuvius".lower()
    ],

    # Temperature
    "sea_surface_temperature": [
        "sea surface temperature", "sea surface temperatures", "sea zone of a temperature",
        "water temperature"
    ],
    "land_surface_temperature": [
        "land surface temperature", "lst", "daytime temperature",
        "surface temperature", "air temperature",
        "temperature",
        "temperature at 2 meters", "air temperature at 2m", "fahrenheit", "celcius", "hotter"
    ],

     # Fire
    "fire_smoke_detection": [
        "active fire", "fire", "smoke plume","smoke", "burn scar", "burned area", "area burned"
    ],

    # Urban & Human Activity
    "urban_human_activity": [
        "urban area", "aritifical land", "built-up area", "city region increase", "nighttime", "nightime",
        "land use", "solar farm", "solar panels"
    ],

     # Land Cover
    "forest_cover": [ # change in presence of trees
        "forest", "deforestation", "tree cover"
    ],
    "vegetation_greenness_crops": [
        "vegetation", "greenness", "ndvi", "crop", "grasslands", "greener", 
        "leaf area", "more yellow areas", "brown/orange", "brown fields",
        "agricultural fields", "red flowers", "brown"
    ],
    
    # Hydrosphere (water/ice)
     "water_color_sediment": [
        "water redness", "sediment", "chlorophyll", "phytoplankton", "algae bloom", "milky", "larger bloom", "ocean surface color", "turbudity", "turbidity"
    ],
    "water_surface_extent": [
        "lake", "reservoir", "river", "wetland", "open water",
        "water surface area", "surface water extent", "flood", 
        "inundat", "water extent", "water storage", "higher water height", "groundwater",
        "lac rouge", "surface water"
    ],
    "snow_ice_cover": [
        "snow", "snowpack", "snow cover", "sea ice", "glacier",
        "ice sheet", "ice cover", "ndsi"
    ],
    "precipitation": [
        "rainfall", "precipitation", "inches of rain", "cm of rain", "less rain", "millimeters of rain"
    ],
    "soil_moisture": [
        "soil moisture", "more moist"
    ],
    "evapotranspiration": [
        "evapotranspiration"
    ],
}

SUBCATEGORY_ORDER = list(SUBCATEGORY_KEYWORDS.keys())

def assign_subcategory(question: str) -> str:
    if not isinstance(question, str):
        return "uncategorized"

    q = question.lower()
    for subcat in SUBCATEGORY_ORDER:
        for kw in SUBCATEGORY_KEYWORDS[subcat]:
            if kw in q:
                return subcat

    return "uncategorized"

def tag_csv(
    input_path: str,
    output_path: str,
    question_col: str = "Question",
    tag_col: str = "Tag",
):
    df = pd.read_csv(input_path)

    if question_col not in df.columns:
        raise

    df[tag_col] = df[question_col].apply(assign_subcategory)
    df.to_csv(output_path, index=False)
    print(f"Wrote tagged CSV to {output_path}")

In [2]:
tag_csv(input_path="./dataset/2026_acl_univearth_1123.csv",
        output_path="2026_acl_univearth_1123_tagged_v2.csv",
        question_col="Question",
        tag_col="Tag")

Wrote tagged CSV to 2026_acl_univearth_1123_tagged_v2.csv


In [26]:
import plotly.express as px
import plotly.io as pio

df = pd.read_csv("2026_acl_univearth_1123_tagged_v2.csv")

SUBCATEGORY_TO_PARENT = {
    "water_surface_extent": "Hydrosphere",
    "water_color_sediment": "Hydrosphere",
    "snow_ice_cover": "Hydrosphere",
    "precipitation": "Hydrosphere",
    "soil_moisture": "Hydrosphere",
    "evapotranspiration": "Hydrosphere",

    "trace_gasses": "Atmosphere",
    "aerosals": "Atmosphere",
    "clouds_fog": "Atmosphere",

    "land_surface_temperature": "Temperature",
    "sea_surface_temperature": "Temperature",

    "fire_smoke_detection": "Fire",

    "vegetation_greenness_crops": "Land Cover",
    "forest_cover": "Land Cover",

    "urban_human_activity": "Human",

    "volcano": "Geology",
    "landform_change": "Geology",
}

df["Parent"] = df["Tag"].map(SUBCATEGORY_TO_PARENT)


def make_tag_label(tag: str) -> str:
    if tag == "urban_human_activity":
        base = "Urban & <br>Human Activity"
    elif tag == "fire_smoke_detection":
        base = "Fire &<br>  Smoke"
    elif tag == "vegetation_greenness_crops":
        base = "Vegetation  <br>Condition"
    elif tag == "water_color_sediment":
        base = "Water Color"
    elif tag == "clouds_fog":
        base = "Clouds & Fog"
    elif tag == "snow_ice_cover":
        base = "Snow & Ice"
    elif tag == "water_surface_extent":
        base = "Water Surface"
    elif tag == "land_surface_temperature":
        base = "Land"
    elif tag == "sea_surface_temperature":
        base = "Sea"
    else:
        base = tag.replace("_", " ").title()
    return "  " + base + "  "

df["Tag_Label"] = df["Tag"].apply(make_tag_label)


counts = (
    df.groupby(["Parent", "Tag_Label"])
      .size()
      .reset_index(name="count")
)


fig = px.sunburst(
    counts,
    path=["Parent", "Tag_Label"], 
    values="count",
    branchvalues="total",
    color="Parent",
    color_discrete_sequence=px.colors.qualitative.Set2,
)


fig.update_layout(
    margin=dict(t=10, l=0, r=0, b=10),
)

# Inner layer
fig.update_traces(
    selector=dict(type="sunburst", level=1),
    textfont=dict(size=20),
)

# Outer layer
fig.update_traces(
    selector=dict(type="sunburst", level=2),
    textfont=dict(size=16),
    # uniformtext=dict(minsize=3, mode="hide")
)

fig.update_traces(
    textinfo="label+percent root",
    marker=dict(line=dict(color="white", width=4)),
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Count: %{value}<br>"
        "Parent: %{parent}<extra></extra>"
    )
)

fig.show()

fig.write_image("./figures/sunburst_chart.png", scale=3.1)

In [23]:
import plotly.io as pio
from PIL import Image
import io

# 1. Convert the Plotly figure to a PNG image (in bytes)
#    We use a specific width/height/scale to ensure high quality

# 2. Load the image from bytes using PIL
img = Image.open("./figures/sunburst_chart.png")

# 3. Calculate crop coordinates
#    format: (left_x, top_y, right_x, bottom_y)
width, height = img.size
left_crop = right_crop = 300

coords = (
    left_crop,          # Left (x) start
    0,                  # Top (y) start
    width - right_crop, # Right (x) end
    height              # Bottom (y) end
)

# 4. Crop the image
img_cropped = img.crop(coords)

# 5. Save the result
img_cropped.save("./figures/sunburst_chart_cropped.png")

print(f"Original size: {width}x{height}")
print(f"New size: {img_cropped.size[0]}x{img_cropped.size[1]}")

Original size: 2170x1550
New size: 1570x1550


In [ ]:
# load /Users/ck696/Documents/GitHub/univearth_acl/dataset/2026_acl_univearth_1123_1209_modalities.csv with pandas 
import pandas as pd


In [62]:
from collections import Counter
import pandas as pd
df1 = pd.read_csv("/Users/ck696/Documents/GitHub/univearth_acl/dataset/2026_acl_univearth_1123_1209_modalities-1.csv")
df2 = pd.read_csv("/Users/ck696/Documents/GitHub/univearth_acl/dataset/2026_acl_univearth_1123_1209_modalities-2.csv")
df = pd.concat([df1, df2], ignore_index=True)
df_modalities = df["Modality"].value_counts()
print(f"row counts: {len(df)}")
# Initialize a counter
final_counts = Counter()

# Iterate through the existing index (names) and values (counts)
for label, count in df_modalities.items():
    # value_counts() on a DataFrame often creates a MultiIndex (tuples). 
    # This check ensures we get the string regardless of format.
    if isinstance(label, tuple):
        label = label[0]
        
    # Split the string by newlines
    # str(label) ensures we don't crash if there's a loose Number/NaN
    parts = str(label).split('\n')
    
    for part in parts:
        # Strip removes accidental whitespace around the words
        clean_part = part.strip()
        final_counts[clean_part] += count

# Convert back to a Pandas Series for a clean view
result = pd.Series(final_counts).sort_values(ascending=False)

print(result)

row counts: 449
Landsat 8                              66
VIIRS                                  60
MODIS-Terra                            45
MODIS-Aqua                             40
Landsat 9                              35
                                       ..
News?                                   1
Tropical Rainfall Measuring Mission     1
Sentinel-1                              1
Sentinel-6                              1
Sentinel-5P                             1
Length: 76, dtype: int64


In [ ]:
import pandas as pd
import re

def get_category_and_label(raw_name):
    """
    Returns a tuple: (Category, Display_Label)
    This logic merges specific versions (e.g. Landsat 8/9) into one row 
    to prevent the table from being 40 rows long, while keeping the count accurate.
    """

    if "GRACE-FO" in raw_name:
        raw_name = "GRACE"
    
    n = str(raw_name).lower().strip()

    if any(x.lower() in n for x in ["meris", "misr", "OMPS", "POES", "GOES", "MTSAT", "METEOSAT", "noaa"]):
        return None, None
    
    # --- Category: Global Monitoring ---
    if any(x in n for x in ["modis", "viirs", "seawifs", "meris", "misr"]):
        cat = "Global Monitoring (Coarse Optical)"
        if "modis" in n: return cat, "MODIS System (Terra/Aqua)"
        if "viirs" in n: return cat, "VIIRS"
        if "misr" in n:  return cat, "MISR"
        if "seawifs" in n: return cat, "SeaWiFS"
        if "meris" in n: return cat, "MERIS"
        return cat, raw_name

    # --- Category: Land Imaging ---
    if "landsat" in n or "sentinel 2" in n or "sentinel-2" in n:
        cat = "Multispectral Optical Imaging"
        if "landsat" in n: return cat, "Landsat Series (4--9)"
        if "sentinel" in n: return cat, "Sentinel-2"
        return cat, raw_name

    # --- Category: Atmosphere ---
    if any(x in n for x in ["sentinel 5p", "sentinel-5p", "omi", "toms", "omps", "tempo"]):
        cat = "Atmospheric Composition"
        if "sentinel" in n: return cat, "Sentinel-5P (TROPOMI)"
        if "omi" in n: return cat, "OMI-Aura"
        if "toms" in n: return cat, "TOMS"
        return cat, raw_name.replace("?", "")

    # --- Category: Hydrology ---
    if any(x in n for x in ["gpm", "chirps", "smap", "grace", "grace-fo"]):
        cat = "Hydrology \& Cryosphere"
        if raw_name in ["nasa grace", "grace-fo", "NASA GRACE", "GRACE-FO"]:
            raw_name = "GRACE"
        return cat, raw_name.replace("?", "").replace("News (", "").replace(")", "").strip()

    # --- Category: Weather & Models ---
    if any(x in n for x in ["geos", "era5", "noaa", "goes", "poes", "meteosat", "mtsat"]):
        cat = "Weather Satellites \& Models"
        if "geos" in n: return cat, "GEOS Model"
        if "era5" in n: return cat, "ERA5 Reanalysis"
        return cat, raw_name.replace("?", "")

    # --- Category: Context ---
    if "news" in n:
        cat = "Context \& Ground Truth"
        return cat, None

    return "Other", None

# ==========================================
# 3. PROCESSING: Build the Structured Data
# ==========================================
# We need a nested dictionary: data_tree[Category][Display_Label] = Count
data_tree = {}

total_count = 0
for raw_name, count in result.items():
    category, label = get_category_and_label(raw_name)

    if label == "" or label is None:
        continue
    
    if category not in data_tree:
        data_tree[category] = {}

    if label not in data_tree[category]:
        data_tree[category][label] = 0
        
    data_tree[category][label] += count
    total_count += count

# ==========================================
# 4. LATEX GENERATION
# ==========================================

def generate_hierarchical_latex(tree, total_count):
    lines = []
    
    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(r"\caption{Distribution of Data Sources by Domain}")
    lines.append(r"\label{tab:datasources}")
    
    # We define a custom color for the rows (light blue)
    # Ensure \usepackage{colortbl} and \usepackage{xcolor} are in your main .tex
    lines.append(r"\definecolor{headerblue}{rgb}{0.9, 0.95, 1.0}")
    
    lines.append(r"\resizebox{\columnwidth}{!}{%")
    lines.append(r"\begin{tabular}{lr}")
    lines.append(r"\toprule")
    lines.append(r"\textbf{Category -- Instrument / Source} & \textbf{Count} \\")
    
    # Order of categories for logical flow
    preferred_order = [
        "Global Monitoring (Coarse Optical)",
        "Medium Res. Land (Optical)",
        "Atmospheric Composition",
        "Hydrology \\& Cryosphere",
        "Weather Satellites \\& Models",
        "Context \\& Ground Truth",
        "Other / Unspecified"
    ]
    
    for cat in preferred_order:
        if cat in tree:
            lines.append(r"\midrule")
            # The Hierarchy Header Row
            lines.append(rf"\rowcolor{{headerblue}} \multicolumn{{2}}{{l}}{{\textbf{{\textit{{{cat}}}}}}} \\")
            
            # Sort items within category by count (descending)
            items = tree[cat]
            sorted_items = sorted(items.items(), key=lambda x: x[1], reverse=True)
            
            for label, count in sorted_items:
                lines.append(rf"{label} & {count} \\")

    lines.append(r"\midrule")
    lines.append(rf"\textbf{{Total}} & \textbf{{{total_count}}} \\")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"}") # End resizebox
    lines.append(r"\end{table}")
    
    return "\n".join(lines)

# Run output
total_c = result.sum()
print(generate_hierarchical_latex(data_tree, total_count))

\begin{table}[t]
\centering
\footnotesize
\caption{Distribution of Data Sources by Domain}
\label{tab:datasources}
\definecolor{headerblue}{rgb}{0.9, 0.95, 1.0}
\begin{tabular}{lr}
\toprule
\textbf{Category -- Instrument / Source} & \textbf{Count} \\
\midrule
\rowcolor{headerblue} \multicolumn{2}{l}{\textbf{\textit{Atmospheric Composition}}} \\
Sentinel-5P (TROPOMI) & 16 \\
TOMS & 4 \\
OMI-Aura & 4 \\
TEMPO & 2 \\
\midrule
\rowcolor{headerblue} \multicolumn{2}{l}{\textbf{\textit{Hydrology \& Cryosphere}}} \\
GRACE & 11 \\
GPM & 8 \\
CHIRPS & 7 \\
SMAP & 4 \\
\midrule
\rowcolor{headerblue} \multicolumn{2}{l}{\textbf{\textit{Weather Satellites \& Models}}} \\
GEOS Model & 17 \\
ERA5 Reanalysis & 3 \\
\midrule
\textbf{Total} & \textbf{443} \\
\bottomrule
\end{tabular}
\end{table}


<>:46: SyntaxWarning:

invalid escape sequence '\&'

<>:53: SyntaxWarning:

invalid escape sequence '\&'

<>:60: SyntaxWarning:

invalid escape sequence '\&'

<>:46: SyntaxWarning:

invalid escape sequence '\&'

<>:53: SyntaxWarning:

invalid escape sequence '\&'

<>:60: SyntaxWarning:

invalid escape sequence '\&'

/var/folders/mk/sm66qh5d415_q45h2zny_dr80000gp/T/ipykernel_70478/2731435523.py:46: SyntaxWarning:

invalid escape sequence '\&'

/var/folders/mk/sm66qh5d415_q45h2zny_dr80000gp/T/ipykernel_70478/2731435523.py:53: SyntaxWarning:

invalid escape sequence '\&'

/var/folders/mk/sm66qh5d415_q45h2zny_dr80000gp/T/ipykernel_70478/2731435523.py:60: SyntaxWarning:

invalid escape sequence '\&'

